In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [22]:
df = pd.read_csv('train.txt',sep = ';',header=None, names=['text','emotion'] )

In [23]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [24]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [25]:
unique_emotion = df['emotion'].unique()

In [26]:
unique_emotion

<ArrowStringArray>
['sadness', 'anger', 'love', 'surprise', 'fear', 'joy']
Length: 6, dtype: str

In [27]:
emotion_number = {}
i = 0
for emo in unique_emotion: 
    emotion_number[emo] = i
    i+=1

In [28]:
df['emotion'] = df['emotion'].map(emotion_number)

In [29]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


# 1. To Lower Case

In [32]:
df['text'] = df['text'].apply(lambda x : x.lower())

# 2.Remove punctuation

In [33]:
import string 
def remove_punctuation(txt): 
    return txt.translate(str.maketrans('','',string.punctuation))
df['text'] = df['text'].apply(remove_punctuation)

# 3.Remove numbers

In [35]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit(): 
            new +=i
    return new
df['text'] = df['text'].apply(remove_numbers)

# 4. Remove emojis

In [36]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new
df['text'] = df['text'].apply(remove_emojis)

# 5. Remove Stopwords

In [40]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [59]:
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to C:\Users\Uzair
[nltk_data]     Siddiqui\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to C:\Users\Uzair
[nltk_data]     Siddiqui\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [60]:
stop_words = set(stopwords.words('english'))

In [61]:
len(stop_words)

198

In [62]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [63]:
def remove(txt): 
    words = word_tokenize(txt)
    cleaned = []
    for i in words:
        if not i in stop_words: 
            cleaned.append(i)
    return  ' '.join(cleaned)    

In [64]:
df['text'] = df['text'].apply(remove)

In [65]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [76]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

# 1.Bag of Words

In [80]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

In [81]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [82]:
y_pred_bow = nb_model.predict(X_test_bow)

In [84]:
print(accuracy_score(y_test, y_pred_bow))

0.7678125


In [85]:
y_pred_bow

array([0, 5, 0, ..., 5, 5, 0], shape=(3200,))

In [86]:
y_test

8756     0
4660     5
6095     0
304      5
8241     0
        ..
15578    5
5746     5
6395     5
7624     5
15245    0
Name: emotion, Length: 3200, dtype: int64

# 2.TF-IDF

In [88]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [89]:
nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [90]:
y_pred = nb2_model.predict(X_test_tfidf)

In [91]:
print(accuracy_score(y_test, y_pred))

0.6609375


In [92]:
from sklearn.linear_model import LogisticRegression

In [93]:
logistic_model = LogisticRegression(max_iter=1000)

In [94]:
logistic_model.fit(X_train_tfidf,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [95]:
log_pred = logistic_model.predict(X_test_tfidf)

In [96]:
print(accuracy_score(y_test,log_pred ))

0.8615625
